### Here we'll be downloading and looking at visualizing scRNA-seq data.

First, we'll import the relevant packages; scanpy, and anndata.

In [2]:
import scanpy as sc
import anndata as an
import pandas as pd

Now we'll import some data to get an understanding of how scRNA-seq data is stored, and how to visualize it.

In [49]:
X = an.read_h5ad("/opt/andrew/lupus/lupus.h5ad")
X = an.AnnData(X.raw.X, X.obs, X.raw.var)

X.obs = X.obs.rename(
        {
            "batch_cov": "pool",
            "ind_cov": "patient",
            "cg_cov": "Cell Type",
            "ct_cov": "cell_type_lympho",
            "ind_cov_batch_cov": "Condition",
            "Age": "age",
            "Sex": "sex",
            "pop_cov": "ancestry",
        },
        axis=1,
    )

Let's look at what is contained in an AnnData object, which contains the scRNA-seq data.

In [50]:
print(X)


AnnData object with n_obs × n_vars = 1263676 × 32738
    obs: 'pool', 'patient', 'Processing_Cohort', 'louvain', 'Cell Type', 'cell_type_lympho', 'L3', 'Condition', 'age', 'sex', 'ancestry', 'Status', 'SLE_status'
    var: 'gene_ids', 'feature_types-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0'


So we have identifying labels for each rows (X.obs), and genes themselves (X.var). Let's look at obs first. This is a pandas datafame containing individual cell's information, like which drug it was treated with.

In [51]:
print(type(X.obs))
print(X.obs.reset_index())

<class 'pandas.core.frame.DataFrame'>
                                                     index  \
0        CAAGGCCAGTATCGAA-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0...   
1        CTAACTTCAATGAATG-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0...   
2        AAGTCTGGTCTACCTC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0...   
3        GGCTCGATCGTTGACA-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0...   
4        ACACCGGCACACAGAG-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0...   
...                                                    ...   
1263671  GAATGAACACCGGAAA-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0...   
1263672  TAGCCGGGTACCGAGA-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0...   
1263673  AACTCTTTCCGTAGGC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0...   
1263674  AGCTCCTGTAACGTTC-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0...   
1263675  CTGCGGAGTGATAAGT-1-1-0-0-0-0-0-0-0-0-0-0-0-0-0...   

                                 pool            patient Processing_Cohort  \
0                  dmx_YS-JY-22_pool6             HC-546               4.0   
1                  dmx_YS-JY-22_pool6          1132_1132     

Now let's look at vars - this contains the name of each gene.

In [53]:
print(type(X.var))
print(X.var.index)

<class 'pandas.core.frame.DataFrame'>
Index(['MIR1302-10', 'FAM138A', 'OR4F5', 'RP11-34P13.7', 'RP11-34P13.8',
       'AL627309.1', 'RP11-34P13.14', 'RP11-34P13.9', 'AP006222.2',
       'RP4-669L17.10',
       ...
       'KIR3DL2-1', 'AL590523.1', 'CT476828.1', 'PNRC2-1', 'SRSF10-1',
       'AC145205.1', 'BAGE5', 'CU459201.1', 'AC002321.2', 'AC002321.1'],
      dtype='object', length=32738)


Finally let's look at some the raw data. This is contained in the .X portion of the anndata object. This is a sparse matrix to save space for now - this will be converted to a dense matrix for you as you process the data.

In [55]:
print(type(X.X))
print(X.X[0:50, 0:50])

<class 'scipy.sparse._csr.csr_matrix'>
  (2, 42)	1.0
  (4, 27)	1.0
  (4, 42)	1.0
  (5, 42)	1.0
  (12, 35)	1.0
  (12, 42)	1.0
  (13, 42)	3.0
  (15, 35)	1.0
  (17, 42)	1.0
  (19, 42)	1.0
  (26, 42)	2.0
  (27, 27)	1.0
  (27, 42)	6.0
  (28, 35)	1.0
  (28, 42)	5.0
  (30, 42)	3.0
  (32, 42)	1.0
  (37, 42)	1.0
  (39, 42)	1.0
  (42, 42)	1.0
  (43, 27)	1.0
  (44, 42)	1.0
  (45, 42)	2.0
  (46, 42)	2.0


Now we can proceed with data normalization and preprocessing. Let's only look at the first processing cohort condition, take a subset of those cells and then proceed.

In [70]:
X_ctrl = X[X.obs.Processing_Cohort == '1.0', :]
X_ctrl = X_ctrl[0: 10000, :]

In [71]:
print(X_ctrl)

View of AnnData object with n_obs × n_vars = 10000 × 32738
    obs: 'pool', 'patient', 'Processing_Cohort', 'louvain', 'Cell Type', 'cell_type_lympho', 'L3', 'Condition', 'age', 'sex', 'ancestry', 'Status', 'SLE_status'
    var: 'gene_ids', 'feature_types-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0-0'


Ok now we have about 180k cells. Go ahead and preprocess these, cluster them with leiden clustering, construct a umap, and label the cells. You should broadly see CD4 cells, CD8 cells, NK cells, and monocytes. Once you've labeled those make a umap of these cell labels, and then you're good to go! Base this on the PBMC 3k dataset labeling from scanpy online. Have fun :-)

In [ ]:
# Work goes here.